In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pandas.plotting import register_matplotlib_converters

# 解释性描述数据与可视化
- 数据查看与理解
- 数据清洗
- 分组可视化

## informations_households.csv
informations_households.csv - 包含有关每个家庭的信息，如家庭ID、家庭类型、家庭的总电量消耗等。
数据查看与清洗

In [ ]:
informations_households = pd.read_csv('archive/informations_households.csv')
informations_households

In [ ]:
print(informations_households.nunique())

In [ ]:
print(informations_households['Acorn'].value_counts())

In [ ]:
print(informations_households['Acorn_grouped'].value_counts())

In [ ]:
print(informations_households['file'].value_counts())

In [ ]:
informations_households_filtered = informations_households[(informations_households['Acorn'] != 'ACORN-U') & (informations_households['Acorn'] != 'ACORN-')]
informations_households_filtered

In [ ]:
print(informations_households_filtered['Acorn'].value_counts())

In [ ]:
print(informations_households_filtered['Acorn_grouped'].value_counts())

In [ ]:
valid_lclid = informations_households_filtered['LCLid'].unique()
valid_lclid

In [ ]:
unique_acorn_values = informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Affluent']['Acorn'].unique()
unique_acorn_values

In [ ]:
unique_acorn_values = informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Comfortable']['Acorn'].unique()
unique_acorn_values

In [ ]:
unique_acorn_values = informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Adversity']['Acorn'].unique()
unique_acorn_values

In [ ]:
missing_informations_households_values = informations_households_filtered.isnull().sum()
missing_informations_households_values

## daily_dataset

In [ ]:
daily_dataset = pd.read_csv('archive/daily_dataset.csv')
daily_dataset

In [ ]:
daily_dataset['day'] = pd.to_datetime(daily_dataset['day'])
daily_dataset

In [ ]:
filtered_daily_dataset = daily_dataset[daily_dataset['LCLid'].isin(valid_lclid)]
filtered_daily_dataset

In [ ]:
filtered_daily_dataset.info()

In [ ]:
def plot_column_histogram(df, column_name):
    plt.figure(figsize=(10, 6))
    sns.histplot(df[column_name], kde=True)
    plt.title(f'Histogram of {column_name}')
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
plot_column_histogram(filtered_daily_dataset, 'energy_median')

In [ ]:
columns_all = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_sum', 'energy_min']
columns_without_sum = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_min']
columns_without_min = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_sum']

In [ ]:
def plot_column_histograms(df, column_names):
    plt.figure(figsize=(10, 6))
    for column_name in column_names:
        sns.histplot(df[column_name], kde=True, label=column_name)
    plt.title(f'Histogram of {" vs ".join(column_names)}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.xscale('log')  # 使用对数尺度
    plt.legend()
    plt.show()

In [ ]:
plot_column_histograms(filtered_daily_dataset, columns_all)

In [ ]:
plot_column_histograms(filtered_daily_dataset, columns_without_sum)

In [ ]:
plot_column_histograms(filtered_daily_dataset, columns_without_min)

In [ ]:
def plot_multiple_column_histograms(df, columns):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, column in enumerate(columns):
        sns.histplot(df[column], kde=True, ax=axes[i])
        axes[i].set_title(f'Histogram of {column}')
        axes[i].set_xlabel(column)
        axes[i].set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_multiple_column_histograms(filtered_daily_dataset, columns_all)

In [ ]:
def plot_columns_boxplot(df, columns):
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df[columns])
    plt.xticks(rotation=45)
    plt.title('Box plot of Energy Consumption Statistics')
    plt.show()

In [ ]:
plot_columns_boxplot(filtered_daily_dataset, columns_all)

In [ ]:
plot_columns_boxplot(filtered_daily_dataset, columns_without_sum)

In [ ]:
def plot_column_difference(df, column1, column2):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=column1, y=column2, data=df)
    plt.title(f'Scatter plot of {column1} vs. {column2}')
    plt.xlabel(column1)
    plt.ylabel(column2)
    plt.show()

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_median')

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_max')

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_std')

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_sum')

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_min')

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_median', 'energy_max')

In [ ]:
plot_column_difference(filtered_daily_dataset, 'energy_median', 'energy_count')

In [ ]:
missing_filtered_daily_dataset_values = filtered_daily_dataset.isnull().sum()
missing_filtered_daily_dataset_values

In [ ]:
# 假设我们只关注名为'column_name1'和'column_name2'的列
rows_with_missing_values_specific_columns = filtered_daily_dataset[filtered_daily_dataset[['energy_std']].isnull().any(axis=1)]
rows_with_missing_values_specific_columns

In [ ]:
# 假设我们只关注名为'column_name1'和'column_name2'的列
rows_with_missing_values_specific_columns = filtered_daily_dataset[filtered_daily_dataset[['energy_median', 'energy_mean']].isnull().any(axis=1)]

rows_with_missing_values_specific_columns

In [ ]:

register_matplotlib_converters()

def plot_yearly_energy_trend(df, stat_col='energy_mean'):
    # 设置图形的大小
    plt.figure(figsize=(14, 8))

    # 获取年份列表
    years = df['day'].dt.year.unique()
    for year in sorted(years):
        # 选择当前年份的数据
        df_year = df[df['day'].dt.year == year]

        # 按'day'分组并计算每一天的平均用电量
        daily_avg = df_year.groupby('day')[stat_col].mean()

        # 重置索引，确保能够按时间序列绘图
        daily_avg = daily_avg.reset_index()

        # 绘制当前年份的日平均用电量趋势
        plt.plot(daily_avg['day'], daily_avg[stat_col], label=str(year))

    # 设置图例
    plt.legend(title="Year")

    # 设置标题和坐标轴标签
    plt.title(f'Yearly Trend of Daily Average {stat_col.capitalize()}')
    plt.xlabel('Day of Year')
    plt.ylabel(f'Average {stat_col.capitalize()}')

    # 显示网格
    plt.grid(True)

    # 显示图形
    plt.show()

# 假设filtered_daily_dataset是你的DataFrame
# plot_yearly_energy_trend(filtered_daily_dataset)


In [ ]:
plot_yearly_energy_trend(filtered_daily_dataset)

## halfhourly_dataset

In [ ]:
block_0 = pd.read_csv('archive/halfhourly_dataset/halfhourly_dataset/block_0.csv')
block_0

In [ ]:
block_0.nunique()

In [ ]:
block_0.info()

## hhblock_dataset

In [ ]:
block_0 = pd.read_csv('archive/hhblock_dataset/hhblock_dataset/block_0.csv')
block_0

In [ ]:
block_0.info()

In [ ]:
# 路径到包含CSV文件的文件夹
folder_path = 'F:\wwj\PTUA\Project\\archive\hhblock_dataset\hhblock_dataset'

# 获取文件夹内所有CSV文件的路径
csv_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.csv')]

# 读取每个CSV文件为DataFrame，并存入列表
dataframes = [pd.read_csv(file) for file in csv_files]

# 合并所有DataFrame为一个大的DataFrame
hhblock_big_dataframe = pd.concat(dataframes, ignore_index=True)

In [ ]:
# 查看合并后的DataFrame信息
hhblock_big_dataframe.info()

In [ ]:
hhblock_big_dataframe

In [ ]:
hhblock_big_dataframe['day'] = pd.to_datetime(hhblock_big_dataframe['day'])
hhblock_big_dataframe

In [ ]:
filtered_big_hhblock = hhblock_big_dataframe[hhblock_big_dataframe['LCLid'].isin(valid_lclid)]
filtered_big_hhblock

In [ ]:
filtered_big_hhblock.info()

In [ ]:
missing_filtered_big_hhblock_values = filtered_big_hhblock.isnull().sum()
missing_filtered_big_hhblock_values

In [ ]:
# 包含 hh_0 至 hh_47 列
average_consumption = [filtered_big_hhblock[f'hh_{i}'].mean() for i in range(48)]

In [ ]:
# 绘制折线图
def plot_average_daily_consumption(average_consumption):
    plt.figure(figsize=(10, 6))
    plt.plot(range(48), average_consumption, marker='o', linestyle='-', color='b')
    plt.title('Average Energy Consumption per 30 Minutes')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Energy Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=[f'{i//2:02d}:{i%2*30:02d}' for i in range(0, 48, 3)])
    plt.grid(True)
    plt.show()

In [ ]:
# 假设df是你的DataFrame
# 先排除'id'列
df_numeric = filtered_big_hhblock.iloc[:, 2:]  # 选择从第三列到最后的所有列

# 按年分组求均值
df_yearly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('Y')).mean()

# 按季度分组求均值
df_quarterly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('Q')).mean()

# 按月分组求均值
df_monthly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('M')).mean()

# 获取季度和月份的字符串表示，不考虑年份
# 例如，将"2011Q4"转换为"Q4"，将2012-05转换为"M5"
quarter_str = filtered_big_hhblock['day'].dt.to_period('Q').astype(str).str[-2:]
month_str = filtered_big_hhblock['day'].dt.month.apply(lambda x: f"M{x}")

# 按季度分组求均值，这里使用转换后的季度字符串
df_year_quarterly_mean = df_numeric.groupby(quarter_str).mean()

# 按月份分组求均值，这里使用转换后的月份字符串
df_year_monthly_mean = df_numeric.groupby(month_str).mean()


In [ ]:
df_yearly_mean

In [ ]:
df_quarterly_mean

In [ ]:
df_monthly_mean

In [ ]:
df_year_quarterly_mean

In [ ]:
df_year_monthly_mean

In [ ]:
def plot_average_daily_consumption(df, title_suffix):
    plt.figure(figsize=(14, 8))

    # 创建表示24小时内每半小时间隔的x轴标签列表
    x_labels = [f'{i//2:02d}:{i%2*30:02d}' for i in range(48)]

    for index, row in df.iterrows():
        # 转换index为字符串，适用于任何形式的Period或其他索引类型
        index_str = str(index)

        # 确保row.values中的所有值都是可以绘图的数值类型
        # 这里直接使用row.values[:-1]来排除可能的非数值列
        # 假设最后一列是非数值列，如果不是，请根据实际情况调整
        values = [float(value) for value in row.values if isinstance(value, (int, float))]

        # 绘制每半小时的平均用电量
        plt.plot(x_labels, values, marker='o', linestyle='-', label=index_str)

    plt.legend(title="Time Period", loc='upper right')
    plt.title(f'Average Daily Electricity Consumption per 30 Minutes - {title_suffix}')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Electricity Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=x_labels[::3])
    plt.grid(True)
    plt.show()


In [ ]:
plot_average_daily_consumption(df_yearly_mean, 'Yearly')

In [ ]:
plot_average_daily_consumption(df_quarterly_mean, 'Quarterly')

In [ ]:
plot_average_daily_consumption(df_monthly_mean, 'Monthly')

In [ ]:
plot_average_daily_consumption(df_year_quarterly_mean, 'Yearly Quarterly')

In [ ]:
plot_average_daily_consumption(df_year_monthly_mean, 'Yearly Monthly')

In [ ]:

def plot_with_seasonal_colors(df, title_suffix):
    plt.figure(figsize=(14, 8))

    # 定义季节或月份颜色映射
    # 直接通过字符串切片提取出所有唯一的季节/月份
    unique_seasons = sorted({str(index)[-2:] for index in df.index})
    season_colors = plt.cm.viridis(np.linspace(0, 1, len(unique_seasons)))
    color_map = dict(zip(unique_seasons, season_colors))

    # x轴标签
    x_labels = [f'{i//2:02d}:{i%2*30:02d}' for i in range(48)]

    # 遍历DataFrame的每一行
    for index, row in df.iterrows():
        year, season = str(index)[:-2], str(index)[-2:]
        # 确保row.values中的所有值都是可以绘图的数值类型
        values = [float(value) for value in row.values if isinstance(value, (int, float))]

        # 为这个季节/月份的行选择颜色
        color = color_map[season]

        # 绘制每半小时的平均用电量
        plt.plot(x_labels, values, marker='o', linestyle='-', label=f'{year} {season}', color=color)

    # 自定义图例和图表布局
    plt.legend(title="Time Period", loc='upper right', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.title(f'Average Daily Electricity Consumption per 30 Minutes - {title_suffix}')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Electricity Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=x_labels[::3])
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_with_seasonal_colors(df_quarterly_mean, 'Quarterly')

In [ ]:
plot_with_seasonal_colors(df_monthly_mean, 'Monthly')

In [ ]:
uk_bank_holidays = pd.read_csv('archive/uk_bank_holidays.csv')
uk_bank_holidays

In [ ]:
# 首先将uk_bank_holidays的日期列转换为日期格式
uk_bank_holidays['Bank holidays'] = pd.to_datetime(uk_bank_holidays['Bank holidays'])
# 创建一个集合，包含所有的假期日期
holidays_set = set(uk_bank_holidays['Bank holidays'])


In [ ]:
# 使用.copy()确保filtered_big_hhblock是一个独立的副本，避免后续修改影响原始数据
filtered_big_hhblock_copy = filtered_big_hhblock.copy()

# 创建假期标志列，安全地使用.loc[]
filtered_big_hhblock_copy.loc[:, 'is_holiday'] = filtered_big_hhblock_copy['day'].isin(holidays_set)

# 接下来的操作使用filtered_big_hhblock_copy来保证不会触发警告
df_numeric = filtered_big_hhblock_copy.iloc[:, 2:]  # 选择数值列

# 根据is_holiday列分组求均值
df_holidays_mean = df_numeric.groupby(filtered_big_hhblock_copy['is_holiday']).mean()

# 重命名索引以更清晰地表示数据
df_holidays_mean.index = ['out_holidays', 'in_holidays']

df_holidays_mean


In [ ]:
plot_average_daily_consumption(df_holidays_mean.iloc[:,:-1], 'Holidays')

In [ ]:
# 合并DataFrame
merged_info_hhblock_df = pd.merge(filtered_big_hhblock, informations_households_filtered[['LCLid', 'stdorToU', 'Acorn', 'Acorn_grouped']], on='LCLid', how='left')

In [ ]:
merged_info_hhblock_df

In [ ]:
# 选择需要计算均值的列，这里假设从第三列到最后的所有列是用电量数据
df_numeric = merged_info_hhblock_df.iloc[:, 2:-3]  # 从第三列到倒数第二列（最后一列是'stdorToU'）

# 根据'stdorToU'分组计算均值
df_std_tou_mean = df_numeric.groupby(merged_info_hhblock_df['stdorToU']).mean()

# 按'Acorn'分组计算均值
df_acorn_mean = df_numeric.groupby(merged_info_hhblock_df['Acorn']).mean()

# 按'Acorn_grouped'分组计算均值
df_acorn_grouped_mean = df_numeric.groupby(merged_info_hhblock_df['Acorn_grouped']).mean()

In [ ]:
df_std_tou_mean

In [ ]:
df_acorn_mean

In [ ]:
df_acorn_grouped_mean

In [ ]:
plot_average_daily_consumption(df_std_tou_mean, 'Std or ToU')

In [ ]:
plot_average_daily_consumption(df_acorn_mean, 'Acorn')

In [ ]:
plot_average_daily_consumption(df_acorn_grouped_mean, 'Acorn Grouped')

## Weather Data

temperatureMax - 当天的最高气温。
temperatureMaxTime - 最高气温发生的时间。
windBearing - 风向，表示风从哪个方向吹来，通常以角度表示。
icon - 天气情况的图标代码，如晴天、多云等。
dewPoint - 露点温度，表示空气达到饱和（露水开始凝结）的温度。
temperatureMinTime - 最低气温发生的时间。
cloudCover - 云量，表示天空被云层覆盖的比例。
windSpeed - 风速。
pressure - 大气压强。
apparentTemperatureMinTime - 体感最低温度发生的时间。
apparentTemperatureHigh - 当天的最高体感温度。
precipType - 降水类型，如雨、雪。
visibility - 能见度。
humidity - 湿度。
apparentTemperatureHighTime - 最高体感温度发生的时间。
apparentTemperatureLow - 当天的最低体感温度。
apparentTemperatureMax - 当天的最高体感温度。
uvIndex - 紫外线指数。
time - 观测时间。
sunsetTime - 日落时间。
temperatureLow - 当天的最低气温。
temperatureMin - 同temperatureLow，当天的最低气温。
temperatureHigh - 当天的最高气温。
sunriseTime - 日出时间。
temperatureHighTime - 最高气温发生的时间。
uvIndexTime - 紫外线指数达到最高点的时间。
summary - 天气概况的文字描述。
temperatureLowTime - 最低气温发生的时间。
apparentTemperatureMin - 当天的最低体感温度。
apparentTemperatureMaxTime - 最高体感温度发生的时间。
apparentTemperatureLowTime - 最低体感温度发生的时间。
moonPhase - 月相。

In [ ]:
weather_daily_darksky = pd.read_csv('archive/weather_daily_darksky.csv')
weather_daily_darksky

In [ ]:
weather_daily_darksky.info()

In [ ]:
missing_weather_daily_darksky_values = weather_daily_darksky.isnull().sum()
missing_weather_daily_darksky_values

visibility - 能见度，通常以公里或英里为单位，表示在特定天气条件下最远可见的距离。
windBearing - 风向，表示风从哪个方向吹来。通常以角度表示，其中0度代表北风，90度代表东风，180度代表南风，270度代表西风。
temperature - 气温，通常以摄氏度或华氏度为单位，表示空气的热度。
time - 时间，数据采集的具体时间点，通常以UNIX时间戳或可读格式表示。
dewPoint - 露点温度，当空气冷却到无法容纳其中所有水蒸气时，水蒸气凝结成露水的温度。
pressure - 大气压强，表示空气的重量压在地面上的力量，通常以百帕斯卡（hPa）或毫巴（mb）为单位。
apparentTemperature - 体感温度，综合考虑风速、湿度和实际气温对人体感知温度的影响。
windSpeed - 风速，表示风的快慢，通常以每秒米数或每小时英里数表示。
precipType - 降水类型，如雨、雪、冰雹等。
icon - 天气图标的标识，用于直观表示天气状况，如晴天、多云、雨天等。
humidity - 湿度，表示空气中水蒸气的含量，通常以百分比表示。
summary - 天气概况的文字描述，提供对当前或预测天气状况的简短总结。

In [ ]:
import pandas as pd
        
weather_hourly_darksky = pd.read_csv('archive/weather_hourly_darksky.csv')
weather_hourly_darksky

In [ ]:
# 确保time列为datetime格式
weather_hourly_darksky['time'] = pd.to_datetime(weather_hourly_darksky['time'])

In [ ]:
weather_hourly_darksky.info()

In [ ]:
missing_weather_hourly_darksky_values = weather_hourly_darksky.isnull().sum()
missing_weather_hourly_darksky_values

In [ ]:
def plot_weather_boxplot(df, column_name, time_type):
    # 确保time列为datetime格式
    if not pd.api.types.is_datetime64_any_dtype(df['time']):
        df['time'] = pd.to_datetime(df['time'])

    # 根据时间类型提取对应的时间单位
    if time_type == 'hour':
        df['time_unit'] = df['time'].dt.hour
    elif time_type == 'day':
        df['time_unit'] = df['time'].dt.date
    elif time_type == 'month':
        df['time_unit'] = df['time'].dt.month
    elif time_type == 'quarter':
        df['time_unit'] = df['time'].dt.quarter
    elif time_type == 'year':
        df['time_unit'] = df['time'].dt.year
    else:
        raise ValueError("Invalid time_type provided. Choose from 'hour', 'day', 'month', 'quarter', 'year'.")

    # 绘制箱线图
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='time_unit', y=column_name, data=df)
    plt.title(f'Box plot of {column_name} over {time_type}')
    plt.xlabel(time_type.capitalize())
    plt.ylabel(column_name.capitalize())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()


In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'hour')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'hour')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'day')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'day')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'quarter')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'quarter')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'year')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'year')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'visibility', 'hour')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'windBearing', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'dewPoint', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'pressure', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'humidity', 'hour')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'humidity', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'windSpeed', 'hour')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'windSpeed', 'quarter')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'precipType', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'icon', 'month')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'icon', 'quarter')

In [ ]:
plot_weather_boxplot(weather_hourly_darksky, 'summary', 'month')

In [ ]:
def plot_weather_scatterplot(df, column_name1, column_name2):
    # 绘制散点图
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=column_name1, y=column_name2, data=df)
    plt.title(f'Scatter plot of {column_name1} vs. {column_name2}')
    plt.xlabel(column_name1.capitalize())
    plt.ylabel(column_name2.capitalize())
    plt.grid(True)
    plt.show()

In [ ]:
plot_weather_scatterplot(weather_hourly_darksky, 'temperature', 'apparentTemperature')

In [ ]:
def plot_weather_lineplot(df, column_name, time_type):
    # 确保time列为datetime格式
    if not pd.api.types.is_datetime64_any_dtype(df['time']):
        df['time'] = pd.to_datetime(df['time'])

    # 根据时间类型提取对应的时间单位
    if time_type == 'hour':
        df['time_unit'] = df['time'].dt.hour
    elif time_type == 'day':
        df['time_unit'] = df['time'].dt.date
    elif time_type == 'month':
        df['time_unit'] = df['time'].dt.month
    elif time_type == 'quarter':
        df['time_unit'] = df['time'].dt.quarter
    elif time_type == 'year':
        df['time_unit'] = df['time'].dt.year
    else:
        raise ValueError("Invalid time_type provided. Choose from 'day', 'month', 'quarter', 'year'.")

    # 绘制折线图
    plt.figure(figsize=(12, 8))
    sns.lineplot(x='time_unit', y=column_name, data=df)
    plt.title(f'Line plot of {column_name} over {time_type}')
    plt.xlabel(time_type.capitalize())
    plt.ylabel(column_name.capitalize())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'hour')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'hour')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'day')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'day')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'month')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'month')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'quarter')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'quarter')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'year')

In [ ]:
plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'year')

# 时间序列聚类分析
1. 数据预处理
1.1 同步数据时间粒度
家庭用电量数据：以半小时为单位，需要将其转换为小时单位以匹配天气数据的时间粒度。这可以通过取每小时内两个半小时用电量的平均值来实现。
天气数据：已经以小时为单位，直接使用。
1.2 时间对齐
确保两个数据集的时间戳对齐。这可能涉及到将时间戳转换为统一的格式（如UNIX时间戳或标准日期时间格式），并确保两个数据集覆盖相同的时间段。
1.3 数据合并
根据时间戳将家庭用电量数据和天气数据合并为一个数据集。每个时间点的数据应包括用电量和所有天气参数。
2. 特征工程
特征选择：基于数据理解，选择对聚类可能有影响的特征，例如，体感温度、湿度、风速可能会对用电量有直接影响。
特征转换：将所有特征标准化或归一化，以确保它们在相同的尺度上进行比较。
3. 相似性度量与聚类算法选择
相似性度量：考虑使用动态时间弯曲（DTW）作为相似性度量，因为它能够有效处理时间序列之间的时间偏移和伸缩。
聚类算法：可以考虑使用K-均值聚类（特别是如果使用DTW，那么是DTW的变体，如K-Shape聚类），或者层次聚类算法，后者不需要预先指定聚类数目。
4. 聚类执行
执行聚类算法，将时间序列数据分组为不同的聚类。这些聚类可能基于用电行为和天气条件的相似模式。
5. 聚类结果分析与解释
分析聚类结果：检查每个聚类的特征，如平均用电量、平均气温、平均湿度等，以理解不同聚类代表的用电行为和天气条件的模式。
结果可视化：使用时间序列图、雷达图或热图来展示聚类结果，这有助于直观理解不同聚类之间的差异。
6. 后续步骤
根据聚类结果，可以进一步分析特定天气条件下的用电模式，或者识别特定的用电行为模式对应的天气条件，这对于能源需求预测和优化能源供应具有重要意义。


## 数据预处理

1. 家庭用电量数据预处理
1.1 转换时间粒度
由于家庭用电量数据是以半小时为单位，而天气数据是以小时为单位，需要将家庭用电量数据转换为小时单位。这可以通过计算每小时内两个半小时用电量的平均值来实现。

In [ ]:
# 先复制原数据中的非用电量数据列
filtered_big_hhblock_one_hourly = filtered_big_hhblock[['LCLid', 'day']].copy()

# 计算每小时的平均用电量，并将结果添加到新DataFrame中
for i in range(24):
    filtered_big_hhblock_one_hourly[f'hh_{i}'] = filtered_big_hhblock[[f'hh_{2 * i}', f'hh_{2 * i + 1}']].sum(axis=1)

In [ ]:
filtered_big_hhblock_one_hourly

In [ ]:
filtered_big_hhblock_one_hourly.info()

In [ ]:
weather_hourly_darksky.info()

2. 数据合并
2.1 时间格式转换
根据day和time列将两个数据集合并。这可能需要将家庭用电量数据的day列转换为与天气数据中time列相同的日期加小时的格式。

In [ ]:
# # 初始化列表，用于存储每行新数据
# data_list = []
# 
# for index, row in filtered_big_hhblock_one_hourly.iterrows():
#     lclid = row['LCLid']
#     day = row['day']
#     for hour in range(24):
#         # 生成新的 day-hour 列
#         day_hour = pd.to_datetime(f'{day} {hour:02d}:00:00')
#         # 获取当前小时的用电量
#         energy = row[f'hh_{hour}']
#         # 将数据添加到列表中
#         data_list.append({'LCLid': lclid, 'day-hour': day_hour, 'energy': energy})
# 
# # 使用列表创建新的DataFrame
# transformed_df = pd.DataFrame(data_list)
# 
# # 显示结果
# transformed_df


In [ ]:
# # 创建一个空的DataFrame用于存储转换后的格式
# transformed_df = pd.DataFrame(columns=['LCLid', 'day-hour', 'energy'])
# 
# for index, row in filtered_big_hhblock_one_hourly.iterrows():
#     lclid = row['LCLid']
#     day = row['day']
#     for hour in range(24):
#         # 生成新的 day-hour 列
#         day_hour = pd.to_datetime(f'{day} {hour:02d}:00:00')
#         # 获取当前小时的用电量
#         energy = row[f'hh_{hour}']
#         # 添加到新的DataFrame中
#         transformed_df = transformed_df.append({'LCLid': lclid, 'day-hour': day_hour, 'energy': energy}, ignore_index=True)
# 
# # 显示结果
# transformed_df

In [ ]:
# # 假设已经将家庭用电量数据的日期加小时匹配到了天气数据的时间格式
# # 这里简化为day列已经处理好，实际上需要按小时分割和对齐
# 
# # 合并数据
# combined_data = pd.merge(filtered_big_hhblock_one_hourly, weather_hourly_darksky, left_on='day', right_on='time', how='inner')
# combined_data